In [8]:
#export

import whisper
import mlx_whisper
import soundfile as sf
import torch
import subprocess
from pydub import AudioSegment
import os 
import re 
import logging
import warnings
import shutil
import logs 

#import noisereduce as nr
#import librosa

tag="TRANSCRIBE"
''' 
El módulo TRANSCRIBE sirve para hacer transcripciones automáticas de ficheros. Presenta las siguientes 
funcionalidades.
'''

WHISPER_BASE="base"
WHISPER_MEDIUM="medium"
WHISPER_LARGE="large-v2"

MLX_WHISPER_LARGE="large"

In [9]:
#export
# funciones de uso de audio, conversión y troceo de ficheros con pydub. También, mover y eliminar ficheros

def borrar_fichero(root, file_name, file_extension):
    """
    Elimina uun fichero de un directorio específico.
    
    Parámetros:
        root (str): Ruta del directorio donde se encuentra el archivo.
        file_name (str o list): Nombre del archivo o lista de archivos sin extensión.
        file_extension (str): Extensión del archivo (ej: '.txt', '.csv').

    Si el archivo no existe, se genera un mensaje de advertencia en lugar de lanzar una excepción.
    """

    method="borrar_fichero"

    file_path = os.path.join(root, file_name + file_extension)

    if os.path.exists(file_path):
        try:
            os.remove(file_path)
            logging.info(f"{method}- eliminado: {file_path}")
        except Exception as e:
            logging.error(f"{method}-ERROR al eliminar {file_path}: {e}")
    else:
        logging.warning(f"{method}- Archivo no encontrado: {file_path}")

def mover_ficheros(root, ficheros, dir_destino, borrar_transcripcion=True):
    """
    Mueve uno o varios archivos a un directorio destino dentro del root.
    
    Parámetros:
        root (str): Ruta raíz donde se encuentran los archivos.
        ficheros (str o list): Nombre de archivo o lista de archivos a mover. 
            Cuando se recibe una lista, es porque se ha partido el fichero de audio en varios y por tanto, 
            hay varias ficheros de transcripciones.

            Esta variable tiene que venir con la ruta completa, es decir, root ya está incluido.
        dir_destino (str): Nombre del directorio de destino dentro del root.
    """
    method="mover_ficheros"

    dir_proceso=os.path.join(root, dir_destino)
    os.makedirs(dir_proceso, exist_ok=True)  # Asegura que el directorio existe
   
    #Tenemos siempre una lista
    lista_ficheros=[ficheros] if not isinstance(ficheros, list) else ficheros
    mover_transcripcion=False if not isinstance(ficheros, list) else True
    
    logging.debug(f"{method}-START moviendo {len(lista_ficheros)} ficheros {lista_ficheros} → {dir_proceso}")

    for file in lista_ficheros:
        try:
            #dst_path = os.path.join(dir_proceso, file)
            file_path=os.path.join(root, file)
            logging.debug(f"{method}- A mover: {file_path} → {dir_proceso}")
            shutil.move(file_path, dir_proceso)
            logging.debug(f"{method}- Movido: {file_path} → {dir_proceso}")

            if borrar_transcripcion:
                borrar_fichero(root, file_path, ".txt")
            elif mover_transcripcion:
                shutil.move(f"{file_path}.txt", dir_proceso)
                logging.debug(f"{method}- Movido fichero de transcripción: {file_path} → {dir_proceso}")
    
        except Exception as e:
            logging.error(f"{method}-ERROR Se ha produciedo error moviendo {file_path}: {e}")


def unifica_ficheros_txt(root, lista_ficheros, duracion_segmento=-1):
    """
    Une varios archivos de texto en un solo archivo, agregando marcas de tiempo si es necesario.

    Parámetros:
        root (str): Directorio donde se encuentran los archivos de texto.
        lista_ficheros_path (list): Lista de nombres de archivos (sin extensión .txt) a unificar.
        duracion_segmento (int): Duración en minutos de cada segmento. Si es -1, no se añaden marcas de tiempo.

    Retorna:
        str: Ruta del archivo de salida con los textos unificados.
    """

    method="unifica_ficheros_txt"

    #Generamos el nombre del fichero que queremos generar (se elimina el _amplificado, si estuviera, y el _xxx del número de fichero)
    file_texto =re.sub(r'_amplificado', '', lista_ficheros[0])
    file_texto =re.sub(r'_\d{2}', '', file_texto) + ".txt"
    file_texto_output = os.path.join(root, file_texto)
    logging.info(f"{method}-START fichero {file_texto_output}")
    
    #Abrimos el fichero de salida. 
    with open(file_texto_output, 'w') as salida:
        i=0
        for file in lista_ficheros:
            file_origen = os.path.join(root, file)
            file_origen_text = os.path.join(root, f"{file_origen}.txt")

            #Si duración = -1 significa que no está troceado.
            if duracion_segmento!=-1:            
                tiempo_inicio=i* duracion_segmento
                tiempo_final = (i+1) * duracion_segmento  # Calcula el tiempo en minutos

                salida.write(f"[{tiempo_inicio} - {tiempo_final} minutos]\n")

            with open(file_origen_text, 'r') as entrada:
                salida.write(entrada.read() + "\n")  # Agrega el contenido del archivo
            logging.debug(f"{method}- concatenado {file_origen_text}")

            i+=1
    
    return file_texto, file_texto_output

def convierte(root, file_name, file_extension, formato="m4a", formato_destino="mp3"):
    method="convierte"

    logging.info(f"{method}-START convirtiendo {file_name}{file_extension} de formato {formato} a {formato_destino}")

    file_destino=f"{file_name}.{formato_destino}"
    audio_file_path_ori = os.path.join(root, f"{file_name}{file_extension}")
    logging.info(f"{method}- fichero origen: {audio_file_path_ori}")
    audio_file_path_des = os.path.join(root, file_destino)
    logging.info(f"{method}- fichero destino: {audio_file_path_des}")
    
    try:        
        if formato!="wma": 
            logging.info(f"{method}- convirtiendo con dpub")
            audio = AudioSegment.from_file(audio_file_path_ori, format=formato)
            audio.export(audio_file_path_des, format=formato_destino, bitrate="192k") 
        else:
            command = [
            "ffmpeg", "-i", audio_file_path_ori, 
            "-vn", "-ar", "44100", "-ac", "2", "-b:a", "192k",
            audio_file_path_des
            ]
            logging.info(f"{method}- invocando a ffmpeg directamente {command}")
            subprocess.run(command, check=True)
    except Exception as e:
        logging.error(f"{method}-ERROR convrtiendo {audio_file_path_ori}: {e}")

    return file_name, f".{formato_destino}"


def trocea_fichero(root, file_name, file_extension, duracion_segmento=15):
    """
    Divide un archivo de audio en fragmentos más pequeños de duración fija.

    Parámetros:
        root (str): Ruta del directorio donde se encuentra el archivo de audio.
        file_name (str): Nombre del archivo sin la extensión.
        file_extension (str): Extensión del archivo de audio ('.mp3').
        duracion_segmento (int): Duración de cada fragmento en minutos (por defecto 15).

    Retorna:
        list: Lista de archivos generados.
    """

    method="trocea_fichero"
    logging.debug(f"{method}-START {file_name}{file_extension} en fragmentos de {duracion_segmento} minutos.")

    file_path=os.path.join(root, file_name + file_extension)
    audio = AudioSegment.from_mp3(file_path)
    duracion_total = len(audio) / (60 * 1000)  # Convertir a minutos
    logging.debug(f"{method}- fichero de {duracion_total:.2f} minutos.")

    #La duración es el número de minutos que nos pasen en milisegundos. 
    duracion_fragmento=duracion_segmento*60*1000 

    lista_ficheros=[]
    lista_ficheros_path=[]
    for i, inicio in enumerate(range(0, len(audio), duracion_fragmento)):
        #Dividimos el fichero en trozos. 
        fragmento=audio[inicio:inicio + duracion_fragmento]
        
        file_proceso=f"{file_name}_{i:02d}{file_extension}"
        output_file_path = os.path.join(root,  file_proceso)
        
        if not os.path.exists(output_file_path):
            fragmento.export(output_file_path, format="mp3")
            logging.debug(f"{method}- {output_file_path} generado.")
        else:
            logging.debug(f"{method}- {output_file_path} ya existe.")
        
        #Añadimos a la lista
        lista_ficheros.append(file_proceso)
        lista_ficheros_path.append(output_file_path)
        
    return lista_ficheros, lista_ficheros_path



In [10]:
#export
# funciones de uso de audio, trasncripción con whisper.

def transcribe_audio_to_text(root, file, libreria="WHISPER", modelo=WHISPER_BASE, lengua="es", temperature=0, compression_ratio_threshold=2.5, wordtimestamp=True):
    audio_file_path = os.path.join(root, file)
    audio_file_path_txt = os.path.join(root, f"{file}.txt")
    
    if libreria=="WHISPER":
        device = "mps" if torch.backends.mps.is_available() else "cpu"  # Usar Metal en Mac
        #device="cpu"
        # Cargar el modelo Whisper (puedes cambiar 'base' a otro tamaño del modelo si es necesario)
        model = whisper.load_model(modelo)#, device)

        # FORZAR USO DE float32 PARA EVITAR ERRORES EN MPS
        for param in model.parameters():
            param.data = param.data.to(torch.float32)
    
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UserWarning)
            result = model.transcribe(audio_file_path, language=lengua)

    elif libreria=="MLX_WHISPER":
        # Transcribir el audio
        os.environ["MLX_WHISPER_MODEL"] = modelo
        result = mlx_whisper.transcribe(audio_file_path, language=lengua)

    # Guardar la transcripción en un archivo de texto
    with open(audio_file_path_txt, "w") as f:
        f.write(result["text"])
        #for segment in result["segments"]:
        #    f.write(f"[{segment['start']:.2f}s - {segment['end']:.2f}s] {segment['text']}")

def procesa_transcripcion(root, lista_ficheros, libreria="WHISPER", modelo=WHISPER_BASE, duracion_segmento=10):
    method="procesa_transcripcion"

    dir_procesado="procesado"
    logging.debug(f"{method}-START lista_ficheros={lista_ficheros}, libreria={libreria}, modelo={modelo}, duración segmento={duracion_segmento}")
    
    for file in lista_ficheros:
        if not os.path.exists(os.path.join(root, f"{file}.txt")):
            transcribe_audio_to_text(root, file, libreria=libreria, modelo=modelo)
            logging.debug(f"{method}- Transcrito fichero {file} con modelo {modelo}")
        else:
            logging.debug(f"{method}- Fichero {file} ya transcrito")

    #unifica los ficheros generados
    logging.debug(f"{method}- longitud lista --> {len(lista_ficheros)}")
    if len(lista_ficheros) > 1:
        fichero_texto_unificado, fichero_texto_unificado_path=unifica_ficheros_txt(root, lista_ficheros, duracion_segmento)
        logging.debug(f"{method}- Unificado fichero. Resultado={fichero_texto_unificado_path}")

        #Sólo si trocea, mueve los ficheros parciales, y sus transcripciones, al directorio de procesados.
        mover_ficheros(root, lista_ficheros, dir_procesado)
        logging.debug(f"{method}- Ficheros movidos ({lista_ficheros})")
    else:
        fichero_texto_unificado=f"{lista_ficheros[0]}.txt"
        fichero_texto_unificado_path=os.path.join(root, fichero_texto_unificado)
        logging.debug(f"{method}- No se unifica porque el fichero no se ha troceado. Resultado={fichero_texto_unificado_path}")

    #Mueve el fichero de texto unificado al directorio de procesado.
    mover_ficheros(root, fichero_texto_unificado, dir_procesado)
    logging.debug(f"{method}- Fichero movido ({fichero_texto_unificado})")

    fichero_texto_unificado_path = os.path.join(root, f"{dir_procesado}/{fichero_texto_unificado}")
    logging.debug(f"{method}- fichero unificado -> root:{root}, fichero texto:{fichero_texto_unificado}, path_final: {fichero_texto_unificado_path}")

    return fichero_texto_unificado, fichero_texto_unificado_path

In [11]:
#export

def amplifica_voz(root, file, file_extension):
    file_amplificado=f"{file}_amplificado"

    file_origen_path = os.path.join(root, file + file_extension)
    file_amplificado_path = os.path.join(root, file_amplificado + file_extension)
                
    if not os.path.exists(file_amplificado_path):
        comando=f'ffmpeg -i "{file_origen_path}" -af "speechnorm=e=20:r=0.0003:l=1" "{file_amplificado_path}"'
        os.system(comando)
        logging.debug(f" Creado file amplificado con nombre={file_amplificado_path} ")
    else:
        logging.debug(f" El fichero amplificado {file_amplificado_path} ya existe.")

    return file_amplificado


In [12]:
#export 

def genera_transcripcion_fichero(root, file, amplifica=True, elimina_fondo=False, trocea=True, duracion_segmento=28, log_level=logging.INFO):
    method="genera_transcripcion_fichero"

    logger, handler = logs.crea_log(log_level, method)
    logging.info(f"{method}-START Procesando fichero {file} con parametros: amplifica:{amplifica}, trocea:{trocea}, duración segmento:{duracion_segmento}")

    file_name, file_extension = os.path.splitext(file)
    old_extension = file_extension

    file_unificado="-1"
    file_unificado_path="-1"

    if file_extension.upper() in ('.M4A', '.WAV', '.MP3', '.WMA'):
        #No procesamos los ficheros que hemos generado ya, por si es un rearranque.
        if file_name.endswith("_amplificado") or re.search(r"_\d+$", file_name):
            logging.debug(f"{method}- Ignorado {file} por regla de nombres")
                
        #Usamos la variable file_proceso para poder ir cambiando el nombre
        file_convertido=file_name

        #convertimos a .wav o .mp3 para realizar la transcripción
        if file_extension.lower() in ('.m4a', '.wma'):
            file_convertido, file_extension = convierte(root, file_name, file_extension, formato=file_extension[1:4].lower())
            logging.info(f"{method}- -> Convertido fichero. Resultado={file_convertido}")
            
        #Amplificamos la voz 
        file_amplificado=file_convertido
        if amplifica:
            file_amplificado = amplifica_voz(root, file_convertido, file_extension)
            logging.info(f"{method}- -> Amplificado fichero. Resultado={file_amplificado}")

        lista_ficheros = [file_amplificado + file_extension]
        lista_ficheros_path = [os.path.join(root, file_amplificado + file_extension)]
        if trocea:
            #Tenemos que trocear el fichero y transcribir cada trozo.
            lista_ficheros, lista_ficheros_path = trocea_fichero(root, file_amplificado, file_extension, duracion_segmento=duracion_segmento)
            logging.info(f"{method}- -> Ficheros troceados. Generados {len(lista_ficheros)} ficheros.")
        else:
            #No se ha generado segmentos porque no se ha troceado.
            duracion_segmento=-1

        file_unificado, file_unificado_path=procesa_transcripcion(root, lista_ficheros, libreria="WHISPER", modelo=WHISPER_LARGE, duracion_segmento=duracion_segmento)
        logging.info(f"{method}- -> Ficheros transcritos. Unificado en {file_unificado_path}")

        #Mueve el fichero original al directorio de originales.
        file_path=os.path.join(root, file)
        mover_ficheros(root, file_path, "originales")
        logging.info(f"{method}- -> Movido {file_path} a originales.")

        #Si se ha amplificado, elimina el fichero amplificado y convertido.
        if old_extension != file_extension:
            borrar_fichero(root, file_convertido, file_extension)
            logging.info(f"{method}- -> Borrado fichero convertido {file_convertido}.")

        if file_amplificado!=file_name:
            borrar_fichero(root, file_amplificado, file_extension)
            logging.info(f"{method}- -> Borrado fichero amplificado {file_amplificado}.")
    else:
        logging.warning(f"{method}- Fichero {file_name} no procesado.")
        
    logging.info(f"{method}-END ")
    logs.cierra_log(logger, handler)

    return file_unificado, file_unificado_path


In [13]:
#export

def genera_transcripcion(disco_duro, usuario, carpeta, amplifica=True, elimina_fondo=False, trocea=True, duracion_segmento=28, log_level=logging.INFO):
    method="genera_transcripcion"

    logger, handler = logs.crea_log(log_level, method)

    # Crear el path completo a la carpeta de música
    directory = os.path.join(disco_duro, usuario, carpeta)
    logging.info(f"{method}-START Procesando el directorio: {directory}")

    total=0
    n=0

    for root, dirs, files in os.walk(directory):
        #Solo procesamos el directorio, no los subdirectorios.
        del dirs[:]
        logging.info(f"{method}- Encontrados {len(files)} a procesar.")
        n=1

        #Procesamos los ficheros del directorio.
        for file in files:   
            genera_transcripcion_fichero(root, file, amplifica, elimina_fondo, trocea, duracion_segmento, log_level)
            n+=1
            #logging.info(f"{method}- -> {file} convertido.")

    total=total+n
    logging.info(f"{method}-END Total de ficheros -> {total}")

    logs.cierra_log(logger, handler)

In [7]:
usuario = 'Users/zzddfge'
carpeta = 'Dropbox'
#carpeta = 'Personal/Transcribir'
disco_duro = '/Volumes/Macintosh HD'

log_level=logging.INFO
logs.inicia_logs(log_level=log_level)
genera_transcripcion(disco_duro, usuario, carpeta, log_level=log_level, trocea=True, amplifica=False, duracion_segmento=75)

2026-04-17 23:03:50,038 - INFO - genera_transcripcion-START Procesando el directorio: /Volumes/Macintosh HD/Users/zzddfge/Dropbox
2026-04-17 23:03:50,101 - INFO - genera_transcripcion- Encontrados 3 a procesar.
2026-04-17 23:03:50,101 - INFO - genera_transcripcion_fichero-START Procesando fichero .DS_Store con parametros: amplifica:False, trocea:True, duración segmento:75
2026-04-17 23:03:50,101 - WARNING - genera_transcripcion_fichero- Fichero .DS_Store no procesado.
2026-04-17 23:03:50,102 - INFO - genera_transcripcion_fichero-END 
2026-04-17 23:03:50,102 - INFO - genera_transcripcion_fichero-START Procesando fichero Doce hombre sin piedad.mp3 con parametros: amplifica:False, trocea:True, duración segmento:75
2026-04-17 23:04:22,156 - INFO - genera_transcripcion_fichero- -> Ficheros troceados. Generados 2 ficheros.
2026-04-17 23:23:58,325 - INFO - unifica_ficheros_txt-START fichero /Volumes/Macintosh HD/Users/zzddfge/Dropbox/Doce hombre sin piedad.mp3.txt
2026-04-17 23:23:58,326 - IN

In [2]:
usuario = 'Users/zzddfge'
carpeta = 'Dropbox'
disco_duro = '/Volumes/Macintosh HD'

from pydub import AudioSegment
from pydub.playback import play
import os


def extraer_audio_video(video_file, audio_output_file, video_format="mp4", audio_format="mp3"):
    try:
        audio = AudioSegment.from_file(video_file, format=video_format)
        audio.export(audio_output_file, format=audio_format)
    except Exception as e:
        print(f"Error al tratar el fichero de video: {e}")


root=os.path.join(disco_duro, usuario, carpeta)

files=['2024-02-10-12-00-57_0000075-2024_CP_0210120057602', '2024-02-12-11-28-50_0000091-2024_CP_0212112850248'
       , 'Video 1_ 04_07_2024 - 09_46_11', 'Video 1_ 04_07_2024 - 11_39_23', 'Video 1_ 13_02_2024 - 12_39_49'
       , 'Video 1_ 13_02_2024 - 13_18_00']
files=['VID-20250715-WA0012','2022-06-22-12-51-03_0000361-2022_CP_0622125103521']
for video_file in files:
    video_file_path=os.path.join(root, video_file)
    audio_output_file_path=f"{video_file_path}.mp3"
    video_file_path=f"{video_file_path}.mp4"
    extraer_audio_video(video_file_path, audio_output_file_path, video_format="mp4", audio_format="mp3")


In [13]:
usuario = 'Users/zzddfge'
carpeta = 'Personal/Video'
disco_duro = '/Volumes/Macintosh HD'
root=os.path.join(disco_duro, usuario, carpeta)

files=['ch01_20210628093609', 'ch01_20210628100552', 'ch01_20210628103935', 'ch01_20210628111316']
method="convierte audio"

for file in files:
    file_path=os.path.join(root, f"{file}.mp4")
    audio_path=os.path.join(root, f"{file}.mp3")
    command = [
            "ffmpeg", "-i", file_path, 
            "-vn", "-acodec", "libmp3lame", "-b:a", "192K",
            audio_path
    ]
    print(f"{method}- invocando a ffmpeg directamente {command}")
    subprocess.run(command, check=True)
    


convierte audio- invocando a ffmpeg directamente ['ffmpeg', '-i', '/Volumes/Macintosh HD/Users/zzddfge/Personal/Video/ch01_20210628093609.mp4', '-vn', '-acodec', 'libmp3lame', '-b:a', '192K', '/Volumes/Macintosh HD/Users/zzddfge/Personal/Video/ch01_20210628093609.mp3']


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex

convierte audio- invocando a ffmpeg directamente ['ffmpeg', '-i', '/Volumes/Macintosh HD/Users/zzddfge/Personal/Video/ch01_20210628100552.mp4', '-vn', '-acodec', 'libmp3lame', '-b:a', '192K', '/Volumes/Macintosh HD/Users/zzddfge/Personal/Video/ch01_20210628100552.mp3']


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex

convierte audio- invocando a ffmpeg directamente ['ffmpeg', '-i', '/Volumes/Macintosh HD/Users/zzddfge/Personal/Video/ch01_20210628103935.mp4', '-vn', '-acodec', 'libmp3lame', '-b:a', '192K', '/Volumes/Macintosh HD/Users/zzddfge/Personal/Video/ch01_20210628103935.mp3']


Output #0, mp3, to '/Volumes/Macintosh HD/Users/zzddfge/Personal/Video/ch01_20210628103935.mp3':
  Metadata:
    TSSE            : Lavf61.7.100
  Stream #0:0: Audio: mp3, 8000 Hz, mono, s16p, 192 kb/s
      Metadata:
        encoder         : Lavc61.19.101 libmp3lame
[out#0/mp3 @ 0x123e05140] video:0KiB audio:15798KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.009173%
size=   15800KiB time=00:33:42.04 bitrate=  64.0kbits/s speed=1.57e+03x    
ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-libri

convierte audio- invocando a ffmpeg directamente ['ffmpeg', '-i', '/Volumes/Macintosh HD/Users/zzddfge/Personal/Video/ch01_20210628111316.mp4', '-vn', '-acodec', 'libmp3lame', '-b:a', '192K', '/Volumes/Macintosh HD/Users/zzddfge/Personal/Video/ch01_20210628111316.mp3']


[out#0/mp3 @ 0x14e810640] video:0KiB audio:6541KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.022157%
size=    6542KiB time=00:13:57.04 bitrate=  64.0kbits/s speed=1.54e+03x    


In [ ]:
import moviepy.editor as mp

def extraer_audio_video2(video_file, audio_output_file, video_format="mp4", audio_format="mp3"):
    try:
        video = mp.VideoFileClip(video_file)
        video.audio.write_audiofile(audio_output_file, codec=audio_format.lower())
        video.close()
    except Exception as e:
        print(f"Error al tratar el fichero de video: {e}")

root=os.path.join(disco_duro, usuario, carpeta)
video_file="video_prueba.mp4"
audio_output_file="audio_video_prueba.mp3"

video_file_path=os.path.join(root, video_file)
audio_output_file_path=os.path.join(root, audio_output_file)

video_file_path

ModuleNotFoundError: No module named 'moviepy'

In [ ]:
''' 
            if elimina_fondo:
                file_limpio=f"{file_limpio}_limpio.wav"

                file_path = os.path.join(root, file)
                file_limpio_path = os.path.join(root, file_limpio)

                if not os.path.exists(file_amplificado_path):
                    # Cargar el audio original
                    audio, sr = librosa.load(file_path, sr=None)

                    # Reducir ruido
                    audio_limpio = nr.reduce_noise(y=audio, sr=sr)

                    # Guardar el audio limpio
                    sf.write(file_limpio_path, audio_limpio, sr)
                    print(f" - File limpio con nombre={file_limpio_path} ")
                else:
                    print(f" - File limpio ya existe.")
''' 

In [1]:
!pip install -U mlx-audio

  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 16.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.9/584.9 kB 12.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 MB 16.2 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 15.9 MB/s  0:00:00 eta 0:00:01
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
  Attempting uninstall: regex
    Found existing installation: regex 2024.11.6
    Uninstalling regex-2024.11.6:
      Successfully uninstalled regex-2024.11.6
  Attempting uninstall: mlx-metal
    Found existing installation

In [6]:
import os
from mlx_audio.vad import load

usuario = 'Users/zzddfge'
carpeta = 'Downloads'
disco_duro = '/Volumes/Macintosh HD'

fichero="prueba.mp3"
#fichero="prueba.wav"
file=os.path.join(disco_duro, usuario, carpeta, fichero)

model = load("mlx-community/diar_sortformer_4spk-v1-fp32")
result = model.generate(file, threshold=0.5, verbose=True)
print(result.text)

for seg in result.segments:
    print(f"Speaker {seg.speaker}: {seg.start:.2f}s - {seg.end:.2f}s - {seg.text}")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Audio: 603.03s
Trimmed 0.33s leading silence
Features: (1, 80, 60304)
Found 1388 segments with 2 speakers
Processing time: 5.28s
SPEAKER audio 1 0.330 12.720 <NA> <NA> speaker_0 <NA> <NA>
SPEAKER audio 1 12.890 17.360 <NA> <NA> speaker_1 <NA> <NA>
SPEAKER audio 1 21.690 2.640 <NA> <NA> speaker_0 <NA> <NA>
SPEAKER audio 1 24.490 0.160 <NA> <NA> speaker_0 <NA> <NA>
SPEAKER audio 1 36.570 20.480 <NA> <NA> speaker_0 <NA> <NA>
SPEAKER audio 1 56.890 20.720 <NA> <NA> speaker_1 <NA> <NA>
SPEAKER audio 1 77.690 29.200 <NA> <NA> speaker_0 <NA> <NA>
SPEAKER audio 1 106.730 16.560 <NA> <NA> speaker_1 <NA> <NA>
SPEAKER audio 1 123.290 0.080 <NA> <NA> speaker_0 <NA> <NA>
SPEAKER audio 1 123.370 0.400 <NA> <NA> speaker_1 <NA> <NA>
SPEAKER audio 1 123.770 0.080 <NA> <NA> speaker_0 <NA> <NA>
SPEAKER audio 1 123.850 0.160 <NA> <NA> speaker_1 <NA> <NA>
SPEAKER audio 1 124.010 0.080 <NA> <NA> speaker_0 <NA> <NA>
SPEAKER audio 1 124.090 1.280 <NA> <NA> speaker_1 <NA> <NA>
SPEAKER audio 1 125.050 0.080 <NA

AttributeError: 'DiarizationSegment' object has no attribute 'text'

In [8]:
vars(seg)

{'start': 0.33, 'end': 13.05, 'speaker': 0}

In [38]:
from pathlib import Path
from mlx_whisper import transcribe
from mlx_audio.vad import load
import subprocess
from pathlib import Path

def formato_min_seg(segundos):
    minutos = int(segundos // 60)
    segundos_restantes = int(segundos % 60)
    return f"{minutos:02d}:{segundos_restantes:02d}"

def transcribir_con_diarizacion(audio_file):
    """
    Devuelve un texto completo con:
    - speaker
    - timestamps
    - texto transcrito
    """

    # -------------------------
    # DIARIZACIÓN
    # -------------------------
    diar_model = load("mlx-community/diar_sortformer_4spk-v1-fp32")

    diar_result = diar_model.generate(
        str(audio_file),
        threshold=0.5,
        verbose=False
    )

    # -------------------------
    # TRANSCRIPCIÓN
    # -------------------------
    whisper_result = transcribe(str(audio_file), path_or_hf_repo="mlx-community/whisper-large-v3-mlx")

    whisper_segments = whisper_result["segments"]

    # -------------------------
    # ASIGNAR SPEAKERS
    # -------------------------
    salida = []

    for wseg in whisper_segments:

        w_start = wseg["start"]
        w_end = wseg["end"]
        w_text = wseg["text"].strip()

        speaker_detectado = "unknown"

        # Buscar speaker cuyo rango temporal coincida
        for dseg in diar_result.segments:

            d_start = dseg.start
            d_end = dseg.end
            d_speaker = dseg.speaker

            # overlap temporal
            if (
                w_start >= d_start and w_start <= d_end
            ) or (
                w_end >= d_start and w_end <= d_end
            ):

                speaker_detectado = d_speaker
                break

        linea = (f"[{formato_min_seg(w_start)} - {formato_min_seg(w_end)}] "
                 f"{speaker_detectado}: "
                 f"{w_text}")

        salida.append(linea)

    # Texto final
    texto_final = "\n".join(salida)

    return texto_final

def limpiar_audio_ffmpg(input_file, output_file=None):
    input_file = Path(input_file)

    if output_file is None:
        output_file = (
            input_file.parent /
            f"{input_file.stem}_limpia.wav"
        )

    output_file = Path(output_file)

    filtro_audio = (
        "highpass=f=100,"
        "lowpass=f=5500,"
        "volume=8dB,"
        "loudnorm=I=-16:LRA=11:TP=-1.5"
    )

    comando = [
        "ffmpeg",
        "-i", str(input_file),
        "-vn",
        "-af", filtro_audio,
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        "-y",

        str(output_file)

    ]

    resultado = subprocess.run(
        comando,
        capture_output=True,
        text=True
    )

    if resultado.returncode != 0:

        print(resultado.stderr)

        return None

    return str(output_file)

def limpiar_audio_deepfilter(input_file, output_file=None):
    """
    Limpia audio usando DeepFilterNet de forma segura.
    Flujo:
    1. Convierte entrada a WAV mono 48 kHz.
    2. Ejecuta deepFilter.
    3. Convierte salida a WAV mono 16 kHz para Whisper/MLX.

    Requiere:
        brew install ffmpeg
        pip install deepfilternet
    """
    method="limpiar_audio_deepfilter"
    print(f"{method}-START con {input_file}")
    
    input_file = Path(input_file)

    if output_file is None:
        output_file = input_file.parent / f"{input_file.stem}_limpia.wav"
    else:
        output_file = Path(output_file)
    print(f"{method}- generando {output_file} ... ")

    temp_48k = input_file.parent / f"{input_file.stem}_tmp_48k.wav"
    deepfilter_out_dir = input_file.parent 

    # 1) Convertir a WAV mono 48 kHz
    cmd_convert_48k = [
        "ffmpeg",
        "-i", str(input_file),
        "-vn",
        "-ac", "1",
        "-ar", "48000",
        "-c:a", "pcm_s16le",
        "-y",
        str(temp_48k),
    ]

    r = subprocess.run(cmd_convert_48k, capture_output=True, text=True)

    if r.returncode != 0:
        print("ERROR convirtiendo a WAV 48 kHz:")
        print(r.stderr)
        return None
    print(f"{method}- Creado WAV mono 48Khz {temp_48k}")

    # 2) Ejecutar DeepFilterNet
    cmd_deepfilter = [
        "deepFilter",
        str(temp_48k),
        "-o",
        str(deepfilter_out_dir),
    ]

    r = subprocess.run(cmd_deepfilter, capture_output=True, text=True)

    if r.returncode != 0:
        print("ERROR ejecutando DeepFilterNet:")
        print(r.stderr)
        return None

    # DeepFilterNet suele generar: nombre_DeepFilterNet3.wav
    posibles_salidas = list(deepfilter_out_dir.glob("*DeepFilterNet3.wav"))

    if not posibles_salidas:
        print("No encuentro ningún WAV generado por DeepFilterNet.")
        print("Contenido del directorio:", list(deepfilter_out_dir.iterdir()))
        return None

    enhanced_48k = posibles_salidas[0]
    print(f"{method}- salida optimizada {enhanced_48k}")

    # Limpieza y renombrado

    try:
        temp_48k.unlink()
        output_final_file = input_file.parent / f"{input_file.stem}_limpia.wav"
        enhanced_48k.rename(output_final_file)
        output_file = output_final_file
        print(f"{method}- fichero final {output_file}")
    except Exception as e:
        print(f"{method}-ERROR {e} ")
        output_file = None 

    return str(output_file)

# -------------------------------------------------
# EJEMPLO
# -------------------------------------------------

root = Path.home() / "Downloads"
file_name = "prueba3_00"
file_extension = ".mp3"
audio = root / file_name

#trocea_fichero(root, file_name, file_extension, duracion_segmento=15)

audio = root / f"{file_name}{file_extension}" 
#audio_limpio = limpiar_audio_ffmpg(audio)
audio_limpio = limpiar_audio_deepfilter(audio)
texto = transcribir_con_diarizacion(audio_limpio)

print(texto)

limpiar_audio_deepfilter-START con /Users/zzddfge/Downloads/prueba3_00.mp3
limpiar_audio_deepfilter- generando /Users/zzddfge/Downloads/prueba3_00_limpia.wav ... 
limpiar_audio_deepfilter- Creado WAV mono 48Khz /Users/zzddfge/Downloads/prueba3_00_tmp_48k.wav
limpiar_audio_deepfilter- salida optimizada /Users/zzddfge/Downloads/prueba3_00_tmp_48k_DeepFilterNet3.wav
limpiar_audio_deepfilter- fichero final /Users/zzddfge/Downloads/prueba3_00_limpia.wav


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[00:00 - 00:29] unknown: y al individuo
[00:30 - 00:31] 0: a ver, lo de la matrícula
[00:31 - 00:32] 0: digo que tenéis que hablar
[00:32 - 00:38] 0: y lo de la matrícula, evidentemente
[00:38 - 00:39] 0: estos son cosas buenas
[00:39 - 00:41] 0: pero que no puedo dar la matrícula
[00:41 - 00:43] 0: sabéis que no se pueden dar las matrículas si se deja el saludo
[00:43 - 00:44] 0: ¿no?
[00:44 - 00:47] 0: no es un
[00:47 - 00:49] 0: contrato bínscula
[00:49 - 00:51] 0: oye, yo he tenido las miniaturas
[00:51 - 00:53] 2: de 10 pero bastante
[00:53 - 00:54] 2: y no me ponen matrícula
[00:54 - 00:58] 2: solo porque puede dar una matrícula
[00:58 - 00:59] 0: cada vez que es cliente matriculado
[01:00 - 01:01] 0: entonces claro, no hay tanta
[01:01 - 01:04] 0: yo era el único con 10
[01:04 - 01:05] 2: y tampoco me la ponían
[01:05 - 01:07] 2: en realidad no estamos obligados
[01:07 - 01:10] 0: a mí me gusta darlas, así que espero darlas
[01:10 - 01:11] 0: como dices
[01:11 - 01:13] 0: en mi 

In [37]:
audio_limpio

'/Users/zzddfge/Downloads/prueba3_00_tmp_48k_DeepFilterNet3.wav'